In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util
import torch
import re
from tqdm import tqdm

import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("✅ GPU kullanılıyor")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("⚠️ GPU yok, CPU kullanılıyor")

✅ GPU kullanılıyor
GPU: Tesla T4


In [3]:
df = pd.read_csv("articles_clean.csv")
df.head()

,Year,Title_TR,Abstract_TR,Keywords_TR,title_tr_clean,abstract_tr_clean,keywords_tr_clean,combined_text
0,2020-2021,İnsansız Hava Araçları için Manyetik Rezonans ...,İnsansız Hava Araçları (İHA) çeşitli alanlarda...,"insansız hava aracı (İHA), lityum batarya, man...",i̇nsansız hava araçları için manyetik rezonans...,i̇nsansız hava araçları (i̇ha) çeşitli alanlar...,"insansız hava aracı (i̇ha), lityum batarya, ma...",i̇nsansız hava araçları için manyetik rezonans...
1,2020-2021,İş Modeli Kanvas ve Yakın İş Modellerinin Savu...,Savunma ve havacılık sanayinin rekabetçi koşul...,"savunma ve uçak sanayii, kanvas iş modeli, üni...",i̇ş modeli kanvas ve yakın i̇ş modellerinin sa...,savunma ve havacılık sanayinin rekabetçi koşul...,"savunma ve uçak sanayii, kanvas iş modeli, üni...",i̇ş modeli kanvas ve yakın i̇ş modellerinin sa...
2,2020-2021,Hava Muharebesinde Otonom Savunma Algoritmasın...,"Bu çalışma kapsamında, temel hava muharebesi m...","bire-bir hava muharebesi, kural tabanlı yöntem...",hava muharebesinde otonom savunma algoritmasın...,"bu çalışma kapsamında, temel hava muharebesi m...","bire-bir hava muharebesi, kural tabanlı yöntem...",hava muharebesinde otonom savunma algoritmasın...
3,2020-2021,Modernization Projects,"Bu makalede, Türk Havacılık ve Uzay Sanayii LI...","artırılmış gerçeklik, model tabanlı izleme, iş...",modernization projects,"bu makalede, türk havacılık ve uzay sanayii li...","artırılmış gerçeklik, model tabanlı izleme, iş...","modernization projects bu makalede, türk havac..."
4,2020-2021,Döner Kanatlı Özgün Bir İHA Tasarımı ve Uçuş K...,Bu çalışmada 4 rotorlu döner kanatlı bir İnsan...,"İHA, motor, uçuş kontrol kartı, özgün yazılım.",döner kanatlı özgün bir i̇ha tasarımı ve uçuş ...,bu çalışmada 4 rotorlu döner kanatlı bir i̇nsa...,"i̇ha, motor, uçuş kontrol kartı, özgün yazılım.",döner kanatlı özgün bir i̇ha tasarımı ve uçuş ...


In [4]:
df["combined_text"].iloc[0]

'i̇nsansız hava araçları için manyetik rezonans kuplaj ile şarj i̇stasyonu tasarımı i̇nsansız hava araçları (i̇ha) çeşitli alanlarda kullanılabilen elektronik sistemlerdir. gerçekleştirilen bu çalışmada, i̇ha’da kullanılan lityum bataryalar için manyetik rezonans kuplaj yöntemi kullanılarak bir şarj istasyonu tasarlanmıştır. devre tasarımlarının analizleri, matlab ve pspice programları sayesinde gerçekleştirilmiştir. ayrıca; şarj cihazında kullanılan bobinler, ansys maxwell programı kullanılarak tasarlanmıştır. tasarlanan şarj istasyonu ile 12 cm’den 3w gücündeki lityum bataryalar şarj edilmiştir. tasarlanan sistem ile i̇ha’nın bataryası şarj olurken bile uçmasına imkân sağlanmaktadır. insansız hava aracı (i̇ha), lityum batarya, manyetik rezonanslı kuplaj, kablosuz güç transferi.'

In [5]:
model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(model_name, device=device)

print(f"✅ Model loaded: {model_name}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded: paraphrase-multilingual-MiniLM-L12-v2


In [6]:
# Ağırlıkları belirliyoruz.
W_TITLE = 0.20
W_ABSTRACT = 0.70
W_KEYWORDS = 0.10

def get_embeddings(text):
    """
    Tek bir metin için SBERT embedding üretir.
    Boş veri gelirse 384 boyutlu sıfır vektörü döner.
    """
    if not isinstance(text, str) or text.strip() == "":
        return np.zeros(384, dtype=np.float32)
    return model.encode(text, convert_to_numpy=True, show_progress_bar=False)

# ilerleme çubuğu (tqdm).
tqdm.pandas()

print("SBERT embedding süreci başlatılıyor...")

# Her alan için ayrı ayrı vektör üretiyoruz
df["title_embedding"] = df["title_tr_clean"].progress_apply(get_embeddings)
df["abstract_embedding"] = df["abstract_tr_clean"].progress_apply(get_embeddings)
df["keywords_embedding"] = df["keywords_tr_clean"].progress_apply(get_embeddings)

def combine_embeddings(row):
    """3 ayrı vektörü ağırlıklarına göre çarpar, toplar ve normalize eder."""
    v_title = np.array(row["title_embedding"])
    v_abstract = np.array(row["abstract_embedding"])
    v_keywords = np.array(row["keywords_embedding"])

    # Ağırlıklı Toplam: w1*v1 + w2*v2 + w3*v3
    weighted_vector = (W_TITLE * v_title) + (W_ABSTRACT * v_abstract) + (W_KEYWORDS * v_keywords)

    # L2 Normalizasyon
    norm = np.linalg.norm(weighted_vector)
    if norm > 0:
        weighted_vector = weighted_vector / norm

    return weighted_vector.tolist()

print("\nAğırlıklı vektörler birleştiriliyor ve normalize ediliyor...")

# Vektörleri birleştirip nihai "embedding" sütununu oluşturuyoruz.
df["embedding"] = df.apply(combine_embeddings, axis=1)

print("\nAğırlıklı Embedding işlemi başarıyla tamamlandı.")
print(f"Toplam Kayıt: {len(df)}")
print(f"Yeni Vektör Boyutu: {len(df['embedding'].iloc[0])}")

df.drop(columns=["title_embedding", "abstract_embedding", "keywords_embedding"], inplace=True)

SBERT embedding süreci başlatılıyor...


100%|██████████| 666/666 [00:07<00:00, 84.50it/s]


Ağırlıklı vektörler birleştiriliyor ve normalize ediliyor...

Ağırlıklı Embedding işlemi başarıyla tamamlandı.
Toplam Kayıt: 666
Yeni Vektör Boyutu: 384


In [7]:
df.head()

,Year,Title_TR,Abstract_TR,Keywords_TR,title_tr_clean,abstract_tr_clean,keywords_tr_clean,combined_text,embedding
0,2020-2021,İnsansız Hava Araçları için Manyetik Rezonans ...,İnsansız Hava Araçları (İHA) çeşitli alanlarda...,"insansız hava aracı (İHA), lityum batarya, man...",i̇nsansız hava araçları için manyetik rezonans...,i̇nsansız hava araçları (i̇ha) çeşitli alanlar...,"insansız hava aracı (i̇ha), lityum batarya, ma...",i̇nsansız hava araçları için manyetik rezonans...,"[-0.0650772675871849, 0.143151193857193, -0.04..."
1,2020-2021,İş Modeli Kanvas ve Yakın İş Modellerinin Savu...,Savunma ve havacılık sanayinin rekabetçi koşul...,"savunma ve uçak sanayii, kanvas iş modeli, üni...",i̇ş modeli kanvas ve yakın i̇ş modellerinin sa...,savunma ve havacılık sanayinin rekabetçi koşul...,"savunma ve uçak sanayii, kanvas iş modeli, üni...",i̇ş modeli kanvas ve yakın i̇ş modellerinin sa...,"[0.015851696953177452, 0.037776604294776917, -..."
2,2020-2021,Hava Muharebesinde Otonom Savunma Algoritmasın...,"Bu çalışma kapsamında, temel hava muharebesi m...","bire-bir hava muharebesi, kural tabanlı yöntem...",hava muharebesinde otonom savunma algoritmasın...,"bu çalışma kapsamında, temel hava muharebesi m...","bire-bir hava muharebesi, kural tabanlı yöntem...",hava muharebesinde otonom savunma algoritmasın...,"[-0.09176358580589294, 0.03505117818713188, -0..."
3,2020-2021,Modernization Projects,"Bu makalede, Türk Havacılık ve Uzay Sanayii LI...","artırılmış gerçeklik, model tabanlı izleme, iş...",modernization projects,"bu makalede, türk havacılık ve uzay sanayii li...","artırılmış gerçeklik, model tabanlı izleme, iş...","modernization projects bu makalede, türk havac...","[0.008592396043241024, 0.03571493923664093, -0..."
4,2020-2021,Döner Kanatlı Özgün Bir İHA Tasarımı ve Uçuş K...,Bu çalışmada 4 rotorlu döner kanatlı bir İnsan...,"İHA, motor, uçuş kontrol kartı, özgün yazılım.",döner kanatlı özgün bir i̇ha tasarımı ve uçuş ...,bu çalışmada 4 rotorlu döner kanatlı bir i̇nsa...,"i̇ha, motor, uçuş kontrol kartı, özgün yazılım.",döner kanatlı özgün bir i̇ha tasarımı ve uçuş ...,"[-0.011717955581843853, 0.059530068188905716, ..."


In [8]:
# Metni temizleme
def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()                      # küçük harfe çevirir
    text = re.sub(r"\s+", " ", text)         # fazla boşlukları siler
    text = re.sub(r"[#%&*_=+<>]", "", text)  # belirli özel karakterleri siler
    return text.strip()

# Kullanıcıdan gelen ham veriler
raw_query_title = "Hava Muharebesinde Otonom Savunma Algoritmasının Geliştirilmesi"
raw_query_abstract = (
    "Bu çalışma kapsamında, temel hava muharebesi manevraları kullanılarak "
    "birebir muharebeler için otonom savunma algoritması geliştirilmiştir. "
    "Algoritma, hedef hava aracı ile beklenmedik bir şekilde karşılaşıldığı durumlarda "
    "saldırı üstünlüğünün sağlanması için en uygun muharebe manevrasını seçmeyi sağlamaktadır. "
    "Algoritmanın test edilmesi amacıyla saldıran ve savunan uçaklar için doğrusal olmayan "
    "dinamik modeller kullanılmıştır. Algoritmada manevra seçimi için temel savaş "
    "manevralarını içeren manevra kütüphanesinden uygun manevrayı seçecek kural tabanlı "
    "bir yapı önerilmiştir. MATLAB/Simulink ortamında yapılan benzetim çalışmaları ile "
    "algoritmanın başarımı test edilmiş ve sonuçlar gösterilmiştir."
)
raw_query_keywords = "bire-bir hava muharebesi, kural tabanlı yöntem, temel hava muharebe manevraları."

# Aramayı da ağırlıklı yapıyoruz.
W_TITLE = 0.20
W_ABSTRACT = 0.70
W_KEYWORDS = 0.10

# Her sorgu parçası için ayrı vektör üret.
vec_title = model.encode(clean_text(raw_query_title), convert_to_numpy=True).astype(np.float32)
vec_abstract = model.encode(clean_text(raw_query_abstract), convert_to_numpy=True).astype(np.float32)
vec_keywords = model.encode(clean_text(raw_query_keywords), convert_to_numpy=True).astype(np.float32)

# Vektörleri ağırlıklarıyla çarp ve topla.
query_vector = (W_TITLE * vec_title) + (W_ABSTRACT * vec_abstract) + (W_KEYWORDS * vec_keywords)

# Formülün bozulmaması için Normalize et.
norm = np.linalg.norm(query_vector)
if norm > 0:
    query_vector = query_vector / norm

# Mevcut veri kümesini karşılaştırma için hazırlama.
corpus_embeddings = np.array(df["embedding"].tolist(), dtype=np.float32)

# Kosinüs benzerliği hesaplama.
similarity_scores = util.cos_sim(query_vector, corpus_embeddings)[0]

# Sonuçları raporlama ve benzerlik skoruna göre sıralama.
df["similarity_score"] = similarity_scores.detach().cpu().numpy()
top_results = df.sort_values(by="similarity_score", ascending=False)

# En benzer ilk 5 sonuç.
print("\n--- Özet Ağırlıklı Arama Sonuçları ---")
with pd.option_context('display.max_colwidth', None):
    display(top_results[["Year","Title_TR", "Abstract_TR","similarity_score"]].head(5))

print("\n", "*"*215)

# En az benzer son 5 sonuç.
print("\n--- En Az Benzer Sonuçlar ---")
with pd.option_context('display.max_colwidth', None):
    display(top_results[["Year","Title_TR", "Abstract_TR","similarity_score"]].tail(5))


--- Özet Ağırlıklı Arama Sonuçları ---


,Year,Title_TR,Abstract_TR,similarity_score
2,2020-2021,Hava Muharebesinde Otonom Savunma Algoritmasının Geliştirilmesi,"Bu çalışma kapsamında, temel hava muharebesi manevraları kullanılarak birebir muharebeler için otonom savunma algoritması geliştirilmiştir. Algoritma, hedef hava aracı ile beklenmedik bir şekilde karşılaşıldığı durumlarda saldırı üstünlüğünün sağlanması için en uygun muharebe manevrasını seçmeyi sağlamaktadır. Algoritmanın test edilmesi amacıyla saldıran ve savunan uçaklar için doğrusal olmayan dinamik modeller kullanılmıştır. Algoritmada manevra seçimi için temel savaş manevralarını içeren manevra kütüphanesinden uygun manevrayı seçecek kural tabanlı bir yapı önerilmiştir. MATLAB/Simulink ortamında yapılan benzetim çalışmaları ile algoritmanın başarımı test edilmiş ve sonuçlar gösterilmiştir.",1.000000
478,2023-2024,Hava Muharebesinde Bulanık Mantık ile Karar,"Hava muharebeleri, saldırı ve savunma manevralarının dengeli bir şekilde yapıldığı karmaşık stratejilere dayanmaktadır. Bu projede, hava muharebelerinde otonom karar alma yeteneğini artırmak için oyun teorisi, bulanık mantık ve dinamik programlama gibi ileri düzey tekniklerin birleşiminden oluşan bir algoritma geliştirilmesi hedeflenmektedir. Geliştirilecek olan algoritma, hava muharebelerinde bilinçli kararlar alabilen otonom sistemlerin geliştirilmesine odaklanmaktadır. Bu amaçla, geliştirilen algoritma simülasyon ortamında test edilecek ve başarı oranı ölçülecektir. Algoritmanın performansı, çeşitli simüle edilmiş muharebe senaryoları üzerinde test edilerek değerlendirilecektir. Başarı metrikleri arasında, misyon başarı oranları, kaynak kullanım verimliliği ve değişen ortamlara uygunluk gibi faktörler bulunacaktır. Bu çalışma, hava muharebelerinde otonom sistemlerin karar alma yeteneğini artırmak için kullanılabilecek yeni ve etkili bir yaklaşım sunmayı amaçlamaktadır.",0.867057
186,2021-2022,Otonom Hava Muharebelerinde Bulanık Metodlar Kullanılarak Karar Verme Algoritmalarının Gerçeklenmesi ve Analizi,"Hava muharebelerinde otonominin giderek yaygın- laşmasıyla birlikte görevleri yüksek başarı oranlarıyla gerçekleş- tirecek uçuş algoritmalarının önemi her geçen gün artmaktadır. Bu çalışmada ise bulanık mantık destekli karar verme algoritma- ları ile otonom hava muharebelerinde meydana gelebilecek du- rumlar için bulanık mantık kural tabanı ile MATLAB kullanıla- rak uçuş senaryolarının simülasyon ortamında gerçeklenmesi amaçlanmıştır. Uçakların it dalaşındaki etkinliği değerlendiril- mektedir. Algoritma, hedef uçak hareketine bağlı olarak bulanık mantık kural tabanına göre uygun referans değerlerini üreterek saldırı manevraları gerçekleştirmektedir.",0.854718
658,2023-2024,Yapay Zeka Pilotajı,"Projemizin başlığı `Davranışsal Klonlama Yapay Zeka Pilotajı` olup, asıl amacı savaş uçaklarında karmaşık manevraları gerçekleştirebilecek bir yapay zeka modeli oluşturmaktır. Proje TUSAŞ ortaklığıyla yürütülmektedir. Bu proje kapsamında split-s manevrası bulunmaktadır. Split-s, uçağın baş aşağı olması için bir tarafa yuvarlandığı ve ardından alt tarafı yere bakana kadar yarım döngü gerçekleştirmek üzere daldığı bir manevradır. Bu manevra uçağın önceki yönün tersi yönde gitmesine neden olur. Bu manevra savaş pilotajında yaygın olarak kullanılmaktadır ve gerçek hayattaki savaş senaryolarında çok önemli rollere sahiptir. Projede kullanılan en önemli yapay zeka teknolojisi, uçuş verilerinin dinamiklerini anlamak için önemli olan ve davranış klonlama modeli oluşturabilmek için anlaşılması gereken zaman serisi verilerindeki uzun vadeli bağımlılıkları modelleyebilen bir tür tekrarlayan sinir ağı olan LSTM'dir (Uzun Kısa Süreli Bellek). Ayrıca temel LSTM modeline performansının yükseltilmesi amacıyla dikkat (attention) özelliği eklenmiştir. LSTM-Attention modeli, daha doğru bir tahmin ve manevra gerçekleştirme süreci için uçuş sırasında kritik öneme sahip bazı özelliklere odaklanan ve bu önemli noktalara öncelik veren bir mekanizmaya sahiptir. Öncelikle böyle bir mod


 ***********************************************************************************************************************************************************************************************************************

--- En Az Benzer Sonuçlar ---


,Year,Title_TR,Abstract_TR,similarity_score
396,2023-2024,Alüminyum/Grafit Tozu Eklentili Hibrit Kompozitlerin Yapısal ve Elektriksel Değerlendirilmesi,"Karbon fiber takviyeli kompozit malzemeler yüksek mekanik dayanımları, hafiflikleri ve fiber doğrultusunda yüksek elektrik iletkenlikleri gibi öne çıkan özellikleri ile havacılık alanında sıklıkla tercih edilmektedir. Fakat, kalınlıkları yönünde elektriksel iletkenliklerinin zayıf olması kullanım alanlarını kısıtlamaktadır. Bu çalışmada, havacılık endüstrisinde yapısal uygulamalarda kullanılan karbon fiber takviyeli polimer kompozitlerin, alüminyum, grafit ve karbon fiber tozları kullanılarak katkılandırılması ile kalınlıkları boyunca elektriksel iletkenliği artırılmıştır. Çalışmada, referans olması için herhangi bir katkı malzemesi kullanılmadan vakum torbalama yöntemi ile karbon fiber takviyeli kompozit malzeme üretimi gerçekleştirilmiştir. Elektriksel iletkenliklerin karşılaştırılması amacıyla, çeşitli boyut ve oranlarda alüminyum, grafit ve karbon fiber tozları içeren 10 farklı parametre ile numuneler üretilmiştir. Ardından, numunelerin kalınlıkları boyunca elektriksel iletkenlik ölçümleri gerçekleştirilmiştir. Ayrıca, standartlarına uygun olarak yapılan çekme, üç nokta eğme ve tabakalar arası kayma testleri ile numunelerin mekanik özelliklerindeki değişimler araştırılmıştır. Çalışma sonucunda, matrise yapılan farklı toz takviyelerinin elektriksel iletkenliği arttırdığı ve mekanik özellikleri de etkilediği tespit edilmiştir.",0.058453
527,2023-2024,Karbon Prepreg ve Film Adhesive Malzemelerin Farklı Ortamlara Bağlı Olarak Yaşlanma Özelliklerinin İncelenmesi,"Bu çalışma, karbon prepreg ve film adhesive malzemelerin farklı ortam koşullarında yaşlanma özelliklerini araştırmaktadır. Karbon prepregler, yüksek mukavemet/ağırlık oranları, iyi enerji emme kapasiteleri ve uzun hizmet ömürleri nedeniyle havacılık sektörü başta olmak üzere birçok alanda yaygın olarak kullanılmaktadır. Bu malzemelerin kesim, hazırlık ve üretim sırasında, temiz oda koşullarında çapraz bağlanma reaksiyonlarının hızlanması, mekanik özellikler ve üretim verimliliği üzerinde olumsuz etkilere neden olmaktadır. Çalışmamızda, prepreg ve film yapıştırıcı malzemelerin oda sıcaklığı, 23°C-60%RH ve 30°C-48%RH koşullarında yaşlanma süreçleri incelenmiş ve bu süreçlerin malzemelerin cam geçiş sıcaklıkları (Tg) ve entalpi değişimleri (dH) ve mekanik özellikleri üzerindeki etkileri değerlendirilmiştir. Yaşlanma süresi boyunca Tg değerlerinde genel bir artış, dH değerlerinde ise bir azalma olduğunu görülmektedir. Porozite sonuçlarına bakıldığında yaşlanma süresi arttıkça prepreg malzemelerdeki gözeneklilik oranında artış gözlemlenmiştir. Yaşlanma işlemine tabii tutulmamış numunlerin porosite oranı %0,36 iken, 30°C-48%RH koşullarında 28 gün yaşlandırılan prepreglerin porozite oranı %0,92 olarak ölçülmüştür. Mekanik özelliklere bakıldığında, interlaminar kesme mukavemeti (ILSS) testleri, yaşlanma süresi arttıkça ILSS değerlerinde azalma olduğunu göstermiştir. Oda sıcaklığında 28 gün yaşlandırılan prepreglerin ILSS değeri %2,65 oranında azalırken, 30°C-48%RH koşullarında yaşlandırılan numunelerin ILSS değeri %12.65 düşmüştür. 23°C-60%RH parametresinde yaşlandırılan karbon prepreg numunelerin mekanik özelliklerinde %4.65 azalma görülmüştür. Film adhesive malzemenin Lap Shear sonuçlarına bakıldığında RT'de 12.6%, 23°C-60%RH 'de %14.2, 30°C-48%RH parametresinde 19.2% azalma gözlemlenmiştir.",0.054904
380,2022-2023,Yapıştırıcı ile Birleştirilen Kompozitlerde Farklı Parametrelerin Kayma Dayanımına Etkisinin İncelenmesi,"Bu çalışmada yapıştırıcı ile birleştirilmiş karbon fiber takviyeli kompozit plakaların tek bindirmeli kayma dayanımları yapıştırıcı kalınlığı, kompozit kalınlığı ve fiber yönlenmesi gibi değişen parametrelerle kıyaslanmıştır. Mukavim bir bağlantı elde etmek ve bunun incelenmesi amaçlanmıştır. İlk olarak, bahsedilen bu parametrelerin malzeme mukavemetine olan etkisi hakkında teorik bilgi verilmiş sonrasında ise

In [9]:
df_finally = df[["Year","Title_TR", "Abstract_TR","Keywords_TR","combined_text","embedding"]]

In [10]:
with pd.option_context('display.max_colwidth', None):
    display(df_finally.head())

,Year,Title_TR,Abstract_TR,Keywords_TR,combined_text,embedding
0,2020-2021,İnsansız Hava Araçları için Manyetik Rezonans Kuplaj ile Şarj İstasyonu Tasarımı,"İnsansız Hava Araçları (İHA) çeşitli alanlarda kullanılabilen elektronik sistemlerdir. Gerçekleştirilen bu çalışmada, İHA’da kullanılan lityum bataryalar için manyetik rezonans kuplaj yöntemi kullanılarak bir şarj istasyonu tasarlanmıştır. Devre tasarımlarının analizleri, Matlab ve PSpice programları sayesinde gerçekleştirilmiştir. Ayrıca; şarj cihazında kullanılan bobinler, ANSYS Maxwell programı kullanılarak tasarlanmıştır. Tasarlanan şarj istasyonu ile 12 cm’den 3W gücündeki lityum bataryalar şarj edilmiştir. Tasarlanan sistem ile İHA’nın bataryası şarj olurken bile uçmasına imkân sağlanmaktadır.","insansız hava aracı (İHA), lityum batarya, manyetik rezonanslı kuplaj, kablosuz güç transferi.","i̇nsansız hava araçları için manyetik rezonans kuplaj ile şarj i̇stasyonu tasarımı i̇nsansız hava araçları (i̇ha) çeşitli alanlarda kullanılabilen elektronik sistemlerdir. gerçekleştirilen bu çalışmada, i̇ha’da kullanılan lityum bataryalar için manyetik rezonans kuplaj yöntemi kullanılarak bir şarj istasyonu tasarlanmıştır. devre tasarımlarının analizleri, matlab ve pspice programları sayesinde gerçekleştirilmiştir. ayrıca; şarj cihazında kullanılan bobinler, ansys maxwell programı kullanılarak tasarlanmıştır. tasarlanan şarj istasyonu ile 12 cm’den 3w gücündeki lityum bataryalar şarj edilmiştir. tasarlanan sistem ile i̇ha’nın bataryası şarj olurken bile uçmasına imkân sağlanmaktadır. insansız hava aracı (i̇ha), lityum batarya, manyetik rezonanslı kuplaj, kablosuz güç transferi.","[-0.0650772675871849, 0.143151193857193, -0.042468469589948654, -0.04538925364613533, 0.018382403999567032, -0.04807769134640694, 0.025823924690485, 0.057593196630477905, -0.014091020449995995, 0.060554057359695435, 0.05500785633921623, 0.02027863822877407, 0.022728657349944115, 0.02414495311677456, 0.018502837046980858, 0.048008453100919724, 0.02044694870710373, -0.031032826751470566, 0.0335245318710804, 0.02761090360581875, 0.09847541153430939, 0.026083067059516907, -0.02339266613125801, 0.04073365405201912, -0.06329251080751419, 0.10140906274318695, -0.0004053104785270989, -0.0246559027582407, -0.00010324776667403057, -0.07358325272798538, 0.02350584790110588, 0.08309119939804077, 0.03882218152284622, -0.03933650627732277, 0.021289721131324768, 0.03372842073440552, 0.05433907359838486, -0.07035679370164871, -0.12191329896450043, -0.04358306899666786, -0.03598638251423836, -0.026036327704787254, 0.08459321409463882, -0.03503740206360817, -0.00031326417229138315, 0.03370632603764534, 0.026389557868242264, -0.07073431462049484, 0.047738637775182724, -0.05811610817909241, 0.084576815366745, 0.03782977536320686, 0.01207168772816658, -0.06256160140037537, 0.03164387494325638, 0.01652396097779274, 0.04162842407822609, -0.03191075101494789, -0.07727064192295074, -0.10357331484556198, -0.028043407946825027, 0.0371570885181427, -0.029126552864909172, 0.03455977514386177, -0.013702882453799248, 0.019750723615288734, 0.04891530051827431, -0.029741395264863968, 0.10498581826686859, -0.060556333512067795, -0.07837212085723877, -0.06363683938980103, 0.04425307735800743, -0.04546957090497017, 0.016521567478775978, 0.07651346921920776, 0.020236602053046227, -0.014828366227447987, 0.02983839251101017, -0.0007734451210126281, 0.03808114677667618, -0.08877720683813095, -0.004476530477404594, -0.04797090217471123, 0.0054430728778243065, 0.0534205287694931, 0.009783918969333172, -0.04723876714706421, 0.03610804304480553, -0.018275581300258636, -0.016591377556324005, -0.02989187091588974, -0.06523501127958298, 0.04091707244515419, -0.053484879434108734, -0.043006107211112976, -0.0407378263771534, -0.07586101442575455, -0.012448422610759735, 0.07541728764772415, ...]"
1,2020-2021,İş Modeli Kanvas ve Yakın İş Modellerinin Savunma ve Havacılık Sektörü Firmalarına Uygulanabilirliğinin İncelenmesi,"Savunma ve havacılık sanayinin

In [11]:
# Kaydetme
df_finally.to_pickle("tusas_liftup_sbert_embeddings.pkl")

In [12]:
df_reloaded = pd.read_pickle("tusas_liftup_sbert_embeddings.pkl")
df_reloaded.head()

,Year,Title_TR,Abstract_TR,Keywords_TR,combined_text,embedding
0,2020-2021,İnsansız Hava Araçları için Manyetik Rezonans ...,İnsansız Hava Araçları (İHA) çeşitli alanlarda...,"insansız hava aracı (İHA), lityum batarya, man...",i̇nsansız hava araçları için manyetik rezonans...,"[-0.0650772675871849, 0.143151193857193, -0.04..."
1,2020-2021,İş Modeli Kanvas ve Yakın İş Modellerinin Savu...,Savunma ve havacılık sanayinin rekabetçi koşul...,"savunma ve uçak sanayii, kanvas iş modeli, üni...",i̇ş modeli kanvas ve yakın i̇ş modellerinin sa...,"[0.015851696953177452, 0.037776604294776917, -..."
2,2020-2021,Hava Muharebesinde Otonom Savunma Algoritmasın...,"Bu çalışma kapsamında, temel hava muharebesi m...","bire-bir hava muharebesi, kural tabanlı yöntem...",hava muharebesinde otonom savunma algoritmasın...,"[-0.09176358580589294, 0.03505117818713188, -0..."
3,2020-2021,Modernization Projects,"Bu makalede, Türk Havacılık ve Uzay Sanayii LI...","artırılmış gerçeklik, model tabanlı izleme, iş...","modernization projects bu makalede, türk havac...","[0.008592396043241024, 0.03571493923664093, -0..."
4,2020-2021,Döner Kanatlı Özgün Bir İHA Tasarımı ve Uçuş K...,Bu çalışmada 4 rotorlu döner kanatlı bir İnsan...,"İHA, motor, uçuş kontrol kartı, özgün yazılım.",döner kanatlı özgün bir i̇ha tasarımı ve uçuş ...,"[-0.011717955581843853, 0.059530068188905716, ..."


In [13]:
df_reloaded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 666 entries, 0 to 665
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Year           666 non-null    object
 1   Title_TR       666 non-null    object
 2   Abstract_TR    666 non-null    object
 3   Keywords_TR    666 non-null    object
 4   combined_text  666 non-null    object
 5   embedding      666 non-null    object
dtypes: object(6)
memory usage: 31.3+ KB


In [14]:
# Görselleştirme

# Veri Hazırlığı
embeddings = np.array(df['embedding'].tolist())

# t-SNE ve Kümeleme
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings)

kmeans = KMeans(n_clusters=6, random_state=42)
df['cluster'] = kmeans.fit_predict(embeddings)

# Grafik koordinatlarını ekle
df['x'] = embeddings_2d[:, 0]
df['y'] = embeddings_2d[:, 1]

# Plotly Grafiği Oluşturma
fig = px.scatter(
    df,
    x='x',
    y='y',
    color='cluster',
    hover_name='Title_TR',
    hover_data={
        'x': False,
        'y': False,
        'Year': True,
        'cluster': True
    },
    title='TUSAŞ LIFT UP Projeleri Anlamsal Benzerlik Uzayı (Etkileşimli t-SNE)',
    labels={'cluster': 'Küme No', 'Year': 'Yıl'},
    template='plotly_white',
    color_continuous_scale='Viridis'
)

# Grafik ayarlarını güncelle
fig.update_traces(marker=dict(size=10, opacity=0.7, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(
    dragmode='pan',
    hoverlabel=dict(bgcolor="white", font_size=12, font_family="Arial")
)

fig.show()